# CASS — 01. EDA & 전처리

CICIDS2018 `training-flow.csv`에 대한 탐색적 데이터 분석(EDA)과 전처리 결과를 검증합니다.

**내용:**
1. 원본 데이터 통계
2. 클래스 분포 (attack_flag / attack_step)
3. UDBB 샘플링 결과
4. 전처리 전/후 피처 분포 비교
5. 피처 상관관계 히트맵

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import TRAIN_FILE, ALL_FEATURES, UDBB_COUNTS
from src.data_loader import load_and_sample, preprocess

plt.rcParams['figure.dpi'] = 120
print('Import OK')

## 1. 원본 데이터 기본 통계

In [ ]:
# 원본 전체 로드 (시간 소요 주의)
df_raw = pd.read_csv(TRAIN_FILE, low_memory=False)
df_raw['attack_flag'] = pd.to_numeric(df_raw['attack_flag'], errors='coerce').fillna(0).astype(int)
df_raw['attack_step'] = df_raw['attack_step'].fillna('benign').astype(str).str.strip().str.lower()

print(f'총 행 수     : {len(df_raw):,}')
print(f'피처 수      : {len(ALL_FEATURES)}')
print(f'Benign       : {(df_raw["attack_flag"]==0).sum():,}')
print(f'Attack       : {(df_raw["attack_flag"]==1).sum():,}')
print()
print('attack_step 분포:')
print(df_raw['attack_step'].value_counts().to_string())

## 2. 클래스 분포 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# attack_flag
vc = df_raw['attack_flag'].value_counts()
axes[0].bar(['Benign (0)', 'Attack (1)'], vc.values, color=['#3498DB', '#E74C3C'])
axes[0].set_title('attack_flag 분포 (원본)', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(vc.values):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom')

# attack_step
vc2 = df_raw['attack_step'].value_counts()
colors = ['#3498DB', '#E74C3C', '#F39C12', '#2ECC71', '#9B59B6']
axes[1].bar(vc2.index, vc2.values, color=colors[:len(vc2)])
axes[1].set_title('attack_step 분포 (원본)', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## 3. UDBB 샘플링 결과

In [ ]:
df_sampled = load_and_sample(use_udbb=True)

print('UDBB 샘플링 결과:')
print(df_sampled['attack_step'].value_counts().to_string())
print(f'\n총 행 수: {len(df_sampled):,}')
print(f'Benign:Attack = {(df_sampled["attack_flag"]==0).sum()}:{(df_sampled["attack_flag"]==1).sum()}')

## 4. 전처리 전/후 피처 분포 비교

In [ ]:
X_scaled, feature_names, scaler = preprocess(df_sampled)
df_scaled = pd.DataFrame(X_scaled, columns=feature_names)

# 대표 피처 6개 분포 비교
sample_feats = ['flow duration', 'flow byts/s', 'tot fwd pkts',
                'flow iat mean', 'syn flag cnt', 'init fwd win byts']
sample_feats = [f for f in sample_feats if f in feature_names]

fig, axes = plt.subplots(2, len(sample_feats), figsize=(18, 6))
fig.suptitle('전처리 전/후 피처 분포 비교', fontsize=13, fontweight='bold')

for j, feat in enumerate(sample_feats):
    # 원본
    axes[0, j].hist(df_sampled[feat].replace({float('inf'): float('nan')}).dropna(),
                    bins=50, color='steelblue', alpha=0.7)
    axes[0, j].set_title(f'{feat}\n(raw)', fontsize=9)
    axes[0, j].set_yscale('log')
    # 전처리 후
    axes[1, j].hist(df_scaled[feat], bins=50, color='tomato', alpha=0.7)
    axes[1, j].set_title(f'{feat}\n(scaled)', fontsize=9)

plt.tight_layout()
plt.show()

## 5. 피처 상관관계 히트맵

In [ ]:
corr = df_scaled.corr(method='pearson')

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    corr, annot=True, fmt='.1f', cmap='coolwarm', center=0,
    linewidths=0.3, annot_kws={'size': 7}, ax=ax
)
ax.set_title('CICIDS2018 피처 상관관계 히트맵 (전처리 후)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 고상관 피처 쌍 출력 (|r| > 0.9)
high_corr = (
    corr.abs()
    .unstack()
    .sort_values(ascending=False)
    .drop_duplicates()
)
high_corr = high_corr[(high_corr > 0.9) & (high_corr < 1.0)]
print(f'|r| > 0.9인 피처 쌍 ({len(high_corr)}개):')
print(high_corr.reset_index().rename(columns={'level_0': 'Feature A', 'level_1': 'Feature B', 0: '|r|'}).to_string(index=False))